# AI VERSE — CYRUS Worker (Google Colab)
**"One Universe. Infinite Intelligence."**

This notebook runs the official **CYRUS** model worker on Google Colab GPU runtimes, powered by **Gemma 4 E4B**.
It implements the AI VERSE worker-pull protocol to securely claim queued jobs, perform model inference, and return results to the FastAPI backend.

**Important Architecture Rules:**
- Public Model Identifier: `cyrus` (Clients interact exclusively with `cyrus`; internal model names are never exposed to customers).
- Dedicated Platform: Google Colab.
- Worker ID: `cyrus-colab-1`.

### Step 1: GPU Detection & Hardware Compatibility Verification
Verifies CUDA availability and checks that VRAM is sufficient for Gemma 4 E4B (requires ~8GB in BF16/FP16 or ~3.5GB in 4-bit).

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemError("FATAL: No CUDA GPU detected! Please change runtime type to T4 GPU (Runtime -> Change runtime type -> T4 GPU).")

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
print(f"Detected GPU: {gpu_name}")
print(f"Total VRAM: {total_vram_gb} GB")

# Verification check for Gemma 4 E4B (~4B parameters)
if total_vram_gb < 7.0:
    print(f"NOTE: VRAM ({total_vram_gb} GB) is under 7GB. 4-bit quantization will be applied automatically.")
else:
    print(f"GPU Compatibility Verified: {gpu_name} has sufficient memory for Gemma 4 E4B in FP16/BF16.")

### Step 2: Install Worker Dependencies
Installs `transformers`, `accelerate`, `bitsandbytes`, and `httpx`.

In [ ]:
!pip install -q --upgrade httpx transformers accelerate bitsandbytes sentencepiece

### Step 3: Load Secrets from Google Colab Secrets (Userdata)
Requires:
- `AI_VERSE_API_URL`: Base URL of your deployed AI VERSE FastAPI backend (e.g., `https://api.aiverse.com`)
- `CYRUS_WORKER_TOKEN`: Cryptographic worker token generated from the AI VERSE Admin Dashboard
- `HF_TOKEN` (Optional): Hugging Face access token for gated Gemma checkpoints

In [ ]:
import os
import sys

# Attempt loading from Google Colab userdata secrets
try:
    from google.colab import userdata
    API_URL = userdata.get('AI_VERSE_API_URL') or 'https://ai-verse-backend-impj.onrender.com'
    WORKER_TOKEN = userdata.get('CYRUS_WORKER_TOKEN') or 'wtk_verse_cyrus_colab_default'
    HF_TOKEN = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN')
except Exception:
    API_URL = os.environ.get('AI_VERSE_API_URL', 'https://ai-verse-backend-impj.onrender.com')
    WORKER_TOKEN = os.environ.get('CYRUS_WORKER_TOKEN', 'wtk_verse_cyrus_colab_default')
    HF_TOKEN = os.environ.get('HF_TOKEN')

WORKER_ID = 'cyrus-colab-1'
MODEL_ID = 'CYRUS'  # Public model identifier: cyrus
PROVIDER = 'Google Colab'

# Exact model backing CYRUS
ACTUAL_MODEL_CHECKPOINT = os.environ.get('CYRUS_MODEL_CHECKPOINT', 'google/gemma-4-e4b-it')

print(f'Configured Worker: {WORKER_ID} for Public Model: {MODEL_ID}')
print(f'Target API Gateway: {API_URL}')
print(f'Actual Model Architecture: Gemma 4 E4B ({ACTUAL_MODEL_CHECKPOINT})')


### Step 4: Modular CYRUS Model Inference Adapter (Gemma 4 E4B)
Encapsulates model weights loading and prompt generation. Decouples internal Gemma 4 E4B checkpoints from the public `cyrus` identity.

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

class CyrusGemmaAdapter:
    """
    CYRUS Model Adapter powered by Gemma 4 E4B.
    Supports FP16/BF16 loading on T4 GPU, with automatic 4-bit fallback if memory constrained.
    """
    def __init__(self, model_checkpoint: str = ACTUAL_MODEL_CHECKPOINT, hf_token: str = HF_TOKEN):
        self.checkpoint = model_checkpoint
        self.token = hf_token
        self.tokenizer = None
        self.model = None
        self.generator = None

    def load_model(self):
        print(f"Loading CYRUS (Gemma 4 E4B) from {self.checkpoint}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.checkpoint, token=self.token)
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        
        # Check if 4-bit quantization needed for lower memory
        total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if device == "cuda" else 0
        quant_config = None
        if total_mem_gb < 7.0:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16
            )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.checkpoint,
            token=self.token,
            torch_dtype=torch_dtype,
            quantization_config=quant_config,
            device_map="auto" if device == "cuda" else None
        )
        self.generator = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer
        )
        print("CYRUS (Gemma 4 E4B) successfully loaded into GPU memory.")

    def generate(self, messages: list, max_new_tokens: int = 1024, temperature: float = 0.7) -> dict:
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        start_time = time.time()
        outputs = self.generator(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=max(temperature, 0.01),
            do_sample=True if temperature > 0.0 else False,
            pad_token_id=self.tokenizer.eos_token_id
        )
        latency = (time.time() - start_time) * 1000
        full_text = outputs[0]["generated_text"]
        reply = full_text[len(prompt):].strip()
        prompt_tokens = len(self.tokenizer.encode(prompt))
        completion_tokens = len(self.tokenizer.encode(reply))
        
        return {
            "text": reply,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "latency_ms": latency
        }

# Instantiate and load Gemma 4 E4B
cyrus_engine = CyrusGemmaAdapter()
cyrus_engine.load_model()

### Step 5: Worker Daemon (Registration, Heartbeat & Job Polling Loop)
Connects to AI VERSE backend, registers `cyrus-colab-1`, sends regular telemetry, claims queued CYRUS jobs atomically, executes Gemma 4 E4B inference, and submits output.

In [ ]:
import httpx
import threading

client = httpx.Client(base_url=API_URL, timeout=40.0)
headers = {
    "X-Worker-Token": WORKER_TOKEN,
    "X-Worker-Id": WORKER_ID
}

# 1. Registration handshake
reg_payload = {
    "worker_id": WORKER_ID,
    "model_id": MODEL_ID,
    "provider": PROVIDER,
    "worker_version": "1.0.0"
}
reg_res = client.post("/api/workers/register", json=reg_payload, headers=headers)
print("Registration Response:", reg_res.status_code, reg_res.json())
if reg_res.status_code != 200:
    raise RuntimeError(f"Registration failed: {reg_res.text}")

# 2. Background Heartbeat Thread
stop_event = threading.Event()
current_job_holder = [None]

def heartbeat_loop():
    while not stop_event.is_set():
        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        gpu_mem = int(torch.cuda.get_device_properties(0).total_memory / (1024**2)) if torch.cuda.is_available() else 0
        avail_mem = int(torch.cuda.memory_reserved(0) / (1024**2)) if torch.cuda.is_available() else 0
        status = "busy" if current_job_holder[0] else "online"
        
        payload = {
            "worker_id": WORKER_ID,
            "status": status,
            "gpu_name": gpu_name,
            "gpu_memory": gpu_mem,
            "available_memory": avail_mem,
            "current_job_id": current_job_holder[0]
        }
        try:
            client.post("/api/workers/heartbeat", json=payload, headers=headers)
        except Exception as e:
            print(f"Heartbeat error: {e}")
        time.sleep(15)

hb_thread = threading.Thread(target=heartbeat_loop, daemon=True)
hb_thread.start()
print("Heartbeat daemon running every 15s.")

# 3. Main Polling Loop
print(f"Listening for incoming {MODEL_ID} jobs on {API_URL}...")
try:
    while True:
        try:
            poll_res = client.get("/api/workers/jobs/next", headers=headers)
            if poll_res.status_code == 200:
                job_data = poll_res.json().get("job")
                if job_data:
                    job_id = job_data["job_id"]
                    current_job_holder[0] = job_id
                    print(f"\n[CLAIMED] Job {job_id} | Priority: {job_data.get('priority')}")
                    payload = job_data.get("payload", {})
                    messages = payload.get("messages", [])
                    temp = payload.get("temperature", 0.7)
                    max_tok = payload.get("max_tokens", 1024)
                    
                    try:
                        # Perform actual model inference with Gemma 4 E4B
                        gen_result = cyrus_engine.generate(messages, max_new_tokens=max_tok, temperature=temp)
                        complete_payload = {
                            "result": {
                                "text": gen_result["text"],
                                "finish_reason": "stop",
                                "prompt_tokens": gen_result["prompt_tokens"],
                                "completion_tokens": gen_result["completion_tokens"]
                            },
                            "input_tokens": gen_result["prompt_tokens"],
                            "output_tokens": gen_result["completion_tokens"],
                            "latency_ms": gen_result["latency_ms"]
                        }
                        c_res = client.post(f"/api/workers/jobs/{job_id}/complete", json=complete_payload, headers=headers)
                        print(f"[COMPLETE] Job {job_id} completed in {gen_result['latency_ms']:.1f}ms. Status: {c_res.status_code}")
                    except Exception as inf_err:
                        print(f"[ERROR] Inference failed on job {job_id}: {inf_err}")
                        client.post(f"/api/workers/jobs/{job_id}/fail", json={"error": str(inf_err), "can_retry": True}, headers=headers)
                    
                    current_job_holder[0] = None
            time.sleep(1.5)
        except Exception as poll_err:
            print(f"Polling loop error: {poll_err}")
            time.sleep(3.0)
except KeyboardInterrupt:
    print("Stopping worker gracefully...")
    stop_event.set()